# The Pop Tart Game

https://www.reddit.com/r/puzzles/comments/1o4s2zy/ultra_hard_puzzle_the_pop_tart_game/

A combinatorial game theory puzzle posted by u/not_a_swedish_vegan.

Can be solved by minimax game tree search, with dynamic programming (memoization) for efficiency.

- https://en.wikipedia.org/wiki/Combinatorial_game_theory
- https://en.wikipedia.org/wiki/Minimax
- https://en.wikipedia.org/wiki/Game_tree
- https://en.wikipedia.org/wiki/Memoization

## Search Feasibility

A number of states is given by the number of ways we can distribute the (indistinguishable) pop tarts amongst the (distinguishable) players and the bowl on each turn. This is given by the stars and bars forumla (non-negative version).

https://en.wikipedia.org/wiki/Stars_and_bars_(combinatorics)#Theorem_two

In [ ]:
import math

In [ ]:
bowl = 10
nturns = 5
nplayers = 3

In [ ]:
math.comb(bowl+nplayers, nplayers) * nturns

As we can see, the number of states is relatively small for a computer to search through (though large for a human to do by hand). Furthermore, most of the states aren't even reachable. 

## Game Tree Search

In [ ]:
import functools

In [ ]:
bowl = 10
nturns = 5
nplayers = 3

min_taken_returned = 0


def more_opt(temp_solution, ref_solution, turn):
    return temp_solution[turn % nplayers] > ref_solution[turn % nplayers] \
        or (temp_solution[turn % nplayers] == ref_solution[turn % nplayers] \
            and temp_solution[(turn-1) % nplayers] < ref_solution[(turn-1) % nplayers])

G = dict()

@functools.cache
def rec(players=(0,)*nplayers, turn=0):
    # end game
    if turn >= nturns:
        return players
    opt_solution = (0,) * nplayers
    if sum(players) == bowl:
        return opt_solution
    # steal
    temp_players = list(players)
    temp_players[turn % nplayers] += temp_players[(turn-1) % nplayers]
    temp_players[(turn-1) % nplayers] = 0
    temp_players = tuple(temp_players)
    temp_solution = rec(temp_players, turn+1)
    if more_opt(temp_solution, opt_solution, turn):
        opt_solution = temp_solution
        G[(players, turn)] = (temp_players, turn+1)
    # take from bowl
    for taken in range(min_taken_returned, bowl - sum(players) + 1):
        temp_players = list(players)
        temp_players[turn % nplayers] += taken
        temp_players = tuple(temp_players)
        temp_solution = rec(temp_players, turn+1)
        if more_opt(temp_solution, opt_solution, turn):
            opt_solution = temp_solution
            G[(players, turn)] = (temp_players, turn+1)
    # return to bowl
    for returned in range(min_taken_returned, players[turn % nplayers] + 1):
        temp_players = list(players)
        temp_players[turn % nplayers] -= returned
        temp_players = tuple(temp_players)
        temp_solution = rec(temp_players, turn+1)
        if more_opt(temp_solution, opt_solution, turn):
            opt_solution = temp_solution
            G[(players, turn)] = (temp_players, turn+1)
    return opt_solution

In [ ]:
rec()

## Solution Path

In [ ]:
players = (0,)*nplayers
turn = 0
print(f"Turn {turn}: players={players}, bowl={bowl-sum(players)}")
while (players, turn) in G:
    players, turn = G[(players, turn)]
    print(f"Turn {turn}: players={players}, bowl={bowl-sum(players)}")

Reachable states:

In [ ]:
len(set(G.keys()) | set(G.values()))

## Conclusion

We can change the parameters in this notebook for different scenarions:

- Must a player take/return at least 1 poptart in their action? i.e. passing is not allowed.
  - Note, this does not seem to change the final!
- Is it 5 turns total, or 5 turns each (15 total)?
  - This does change the final result!
- Does the 1 turn each scenario match our expectations?